# 08.3 — Monitor table compatibility

MATLAB's pipeline has a "monitor" system: it watches per-epoch validation metrics, drives **model selection** (keep the best-validation weights as *Optimal*), and keeps a validation results table current so external tooling can read the best model's performance mid-run. This notebook is about how the port reproduces that behavior — and it opens with an honesty note: **there is no `Monitor` class in the Python code.** The `training/monitoring/` module is a stub. The monitoring *functionality* is real, but it lives in a small event system — `EpochHistory.is_best` → an "on optimal" callback → a rewrite of `CM_Table_Validation.mat` — not in a monitor object.

This notebook covers:

* The monitor's job: track validation metrics, select the best model.
* The honest state: `monitoring/` is a stub; the mechanism is a callback.
* `EpochHistory.is_best` and the `OnOptimalCallback` chain.
* Why `CM_Table_Validation.mat` is rewritten on every new best.

**Prerequisites:** [08.2 writing .mat files](08.2_writing_mat_files_for_matlab.ipynb), [05.5 checkpoint state machine](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb).

## Section 1 — What MATLAB does

MATLAB's `trainnet` monitor tracks metrics each epoch and, on a new best, snapshots the Optimal weights and re-saves the validation table:

```matlab
for epoch = 1:numEpochs
    valAcc = validate(net, valData);
    updateMonitor(monitor, epoch, valAcc);        % live progress display
    if valAcc > bestAcc                            % model selection
        bestAcc = valAcc;
        OptimalWeights = net.Learnables;           % snapshot the best model
        cgg_saveValidationCMTable(net, valData);   % rewrite CM_Table_Validation.mat
    end
end
```

The monitor does two jobs: **display** (a live training plot) and **selection** (track the best and snapshot it). The port keeps the *selection* logic faithfully; the *display* is a stdout print rather than a GUI. The validation table is the durable artifact — rewritten on each new best so it always reflects the Optimal model.

## Section 2 — The Python concepts you need

### 2.1 — The honest state: `monitoring/` is a stub

In [ ]:
from pathlib import Path
mon = Path("../../src/neural_data_decoding/training/monitoring/__init__.py").read_text()
print("training/monitoring/__init__.py contents:")
print(repr(mon))
print()
print("→ a docstring listing PLANNED features (W&B logger, .mat writer, memory probe) — no code.")
print("  'registered ≠ implemented' (03.5's lesson): the module exists, the Monitor class doesn't.")

This is the same honesty point as [03.5](../03_data_pipeline/03.5_normalization_strategies.ipynb) (normalization recipes: registered ≠ implemented) and [08.5](08.5_weights_and_biases_integration.ipynb) (W&B: declared dependency, no code). The `monitoring/` package is a placeholder for a future consolidated telemetry module. Today the monitoring *behavior* is assembled from pieces that live elsewhere — and that's what matters for MATLAB compatibility, so let's follow the real mechanism.

### 2.2 — `EpochHistory` and `is_best`

Each epoch, the lifecycle produces an `EpochHistory` — the train metrics, the validation metrics, and a boolean `is_best`: did this epoch's validation accuracy beat the previous best? That single flag is the model-selection signal.

In [ ]:
from neural_data_decoding.training.lifecycle import EpochHistory
import dataclasses

print("EpochHistory fields:")
for f in dataclasses.fields(EpochHistory):
    print(f"  {f.name}: {f.type}")
print()
print("is_best = 'this epoch's val accuracy beats the previous best' — the selection trigger.")

### 2.3 — The `OnOptimalCallback` chain

The lifecycle fires an `OnOptimalCallback` *only* when `is_best` is true. That callback receives the model with its just-saved Optimal weights, and it does the durable work: rewrite `CM_Table_Validation.mat` and `CM_Table.mat` from the Optimal model ([08.2](08.2_writing_mat_files_for_matlab.ipynb)). Simulate the pattern — the table is rewritten only on improvement:

In [ ]:
# A faithful sketch of the lifecycle's best-tracking + on-optimal firing.
def training_with_monitor(val_accuracies):
    best = float("-inf")
    table_writes = []
    for epoch, val_acc in enumerate(val_accuracies, start=1):
        is_best = val_acc > best                      # EpochHistory.is_best
        marker = ""
        if is_best:
            best = val_acc
            table_writes.append(epoch)                # on_optimal → rewrite the table
            marker = "  ← NEW OPTIMAL: CM_Table_Validation.mat rewritten"
        print(f"epoch {epoch}: val acc {val_acc:.3f}{marker}")
    return table_writes

writes = training_with_monitor([0.40, 0.55, 0.52, 0.61, 0.60, 0.63])
print(f"\ntable rewritten on epochs: {writes}  (only when val accuracy improved)")

The validation table is rewritten on epochs 1, 2, 4, 6 — exactly the epochs where a new best appeared, *not* every epoch. This is the crux of monitor compatibility: an external tool (or a human) reading `CM_Table_Validation.mat` at any moment sees the *current best* model's validation results, because the file is refreshed precisely when — and only when — the Optimal model changes. It mirrors the Current-vs-Optimal checkpoint distinction ([05.5 §2.2](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)): Current updates every epoch, Optimal (and its table) only on improvement.

### 2.4 — What MATLAB's monitor expects from the table

MATLAB's model-selection code reads the validation table to get the best model's accuracy, so the table must:

* be present at the run's leaf path ([08.1](08.1_folder_hierarchy_generation.ipynb)) under the name `CM_Table_Validation.mat`;
* have the full `CM_Table` schema ([08.2 §2.2](08.2_writing_mat_files_for_matlab.ipynb)) — `TrueValue`, `Aggregation_Prediction`, the confidence fields — so accuracy can be recomputed from predictions vs targets;
* reflect the *Optimal* model, not the current epoch — guaranteed by the on-best rewrite (§2.3).

The port satisfies all three: the `on_optimal_callback` writes the schema-complete table to the leaf directory whenever `is_best`. So MATLAB's monitor/aggregator finds exactly what it expects, even though no Python `Monitor` object exists.

### 2.5 — The per-epoch telemetry (the display half)

The monitor's other job — live display — is a plain stdout print each epoch (the `EpochCallback`). It reports train/val loss and accuracy, and (with a curriculum) the current lever values ([07.6 §2.5](../07_dynamic_curriculum/07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb)). It's deliberately minimal — a GUI progress plot is what [08.5](08.5_weights_and_biases_integration.ipynb) (Weights & Biases) would replace it with. For now, the durable, MATLAB-compatible artifact is the table; the print is just for the human watching the run.

## Section 3 — The `neural_data_decoding` implementation

The callback types and the `is_best` flag — the monitor's contract:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/lifecycle.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.strip() == "is_best: bool")
print(f"{i + 1:4} | {src[i]}   # the model-selection flag on EpochHistory")
j0 = next(j for j, line in enumerate(src) if "OnOptimalCallback = " in line)
for k in range(j0 - 1, j0 + 1):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `is_best: bool` on `EpochHistory` — `True` only when this epoch's validation accuracy beats the previous best (`False` whenever there's no validation).
* `OnOptimalCallback = Callable[[nn.Module, EpochHistory], None]` — the hook fired on a new best, receiving the Optimal-weighted model. `cli.py`'s `_on_optimal` implements it: write both CM_Tables, print the "New optimal at epoch N" line.
* `EpochCallback = Callable[[EpochHistory], None]` — the per-epoch telemetry hook (§2.5), fired every epoch regardless of `is_best`.
* The lifecycle owns the `best_metric` tracking and the Optimal checkpoint save ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)); the callbacks are where *your* monitoring (table writes, W&B logging, [08.5](08.5_weights_and_biases_integration.ipynb)) plugs in — a clean extension point, even without a `Monitor` class.

## Section 4 — Hands-on exercises

### Exercise 4.1 — count the table writes

For a validation-accuracy sequence, predict how many times the validation table is rewritten (= number of new bests), then verify.

In [ ]:
# Reveal:
accs = [0.5, 0.5, 0.6, 0.58, 0.62, 0.62, 0.7]
best, writes = float("-inf"), 0
for a in accs:
    if a > best:
        best, writes = a, writes + 1
print(f"val accuracies: {accs}")
print(f"table rewrites (new bests): {writes}  — at values {[a for i,a in enumerate(accs) if a > max(accs[:i], default=float('-inf'))]}")
print("→ ties do NOT count as new bests (strict >), so 0.5→0.5 and 0.62→0.62 don't rewrite.")

### Exercise 4.2 — plug in your own monitor

Write an `EpochCallback` that logs each epoch to a list (a stand-in for W&B), showing the callback hook is where custom monitoring attaches — no `Monitor` class needed.

In [ ]:
# Reveal:
log = []
def my_epoch_callback(epoch, train_acc, val_acc, is_best):
    log.append({"epoch": epoch, "train_acc": train_acc, "val_acc": val_acc, "optimal": is_best})

# Simulate three epochs:
for e, (ta, va, best) in enumerate([(0.6, 0.5, True), (0.7, 0.55, True), (0.75, 0.54, False)], start=1):
    my_epoch_callback(e, ta, va, best)
for row in log:
    print(row)
print("→ this is exactly where an 08.5 W&B logger would hook in — the callback IS the monitor API.")

## Section 5 — Common errors

### Looking for a `Monitor` class

There isn't one (§2.1). The `monitoring/` module is a stub; monitoring is the callback chain. Don't import a class that doesn't exist — attach to `EpochCallback` / `OnOptimalCallback`.

### The validation table is stale

It's rewritten only on a *new best* (§2.3), by design — so if the last few epochs didn't improve, the table reflects the earlier Optimal epoch, which is correct. If it's stale in a *wrong* way (never updates), the `on_optimal_callback` isn't wired, or `is_best` never fires (check validation is actually running).

### The table reflects the current epoch, not the best

The write must use the *Optimal* model, not the current one (§2.4). The lifecycle passes the Optimal-weighted model to `on_optimal_callback`; writing from the live model instead would put non-best results in the table.

### Ties rewrite the table

They shouldn't — selection is strict `>` (Exercise 4.1). A `>=` comparison would rewrite on every tie, thrashing the file. Match the strict-improvement semantics.

### Expecting a live GUI plot

The display half is stdout (§2.5), not a plot window. For live curves, wire a W&B callback ([08.5](08.5_weights_and_biases_integration.ipynb)) — the hook is already there.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/lifecycle.py`](../../src/neural_data_decoding/training/lifecycle.py) — `EpochHistory`, `OnOptimalCallback`, the best-tracking loop.
- [08.2 writing .mat files](08.2_writing_mat_files_for_matlab.ipynb) — the schema of the table the monitor rewrites.
- [08.5 Weights & Biases integration](08.5_weights_and_biases_integration.ipynb) — the modern monitor that plugs into these callbacks.

Next up: **[08.4 the .mat round-trip test](08.4_the_mat_round_trip_test.ipynb)** — the T4 parity gate that proves the output files load back correctly.